In [1]:
#%pip install --upgrade pip
#%pip install xgboost catboost lightgbm 

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB 
from sklearn.metrics import accuracy_score 

In [3]:
import sys
import os

sys.path.append(os.path.abspath('..'))

import importlib
import preprocessing

importlib.reload(preprocessing)

from preprocessing import train_clean, test_clean

In [4]:
X = train_clean.drop(columns = ['Survived'])
y = train_clean['Survived']

n_spl = 5
skf = StratifiedKFold(n_splits = n_spl, shuffle = True, random_state = 42)

In [5]:
train_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Survived         891 non-null    int64  
 1   Sex              891 non-null    int64  
 2   Age              891 non-null    float64
 3   Family_Size      891 non-null    int64  
 4   Alone            891 non-null    int64  
 5   Group_Size       891 non-null    int64  
 6   Fare_Per_Person  891 non-null    float64
 7   Fare_log         891 non-null    float64
 8   Group_Survival   891 non-null    float64
 9   Embarked_Q       891 non-null    int64  
 10  Embarked_S       891 non-null    int64  
 11  Title_Miss       891 non-null    int64  
 12  Title_Mr         891 non-null    int64  
 13  Title_Mrs        891 non-null    int64  
 14  Title_Other      891 non-null    int64  
 15  Deck_B           891 non-null    int64  
 16  Deck_C           891 non-null    int64  
 17  Deck_D           891 non-nu

In [6]:
#Linear regression
fold_accuracy = {
    'Linear Regression': [],
    'Ridge (L2)': [],
    'Lasso (L1)': [],
    'ElasticNet (L1 + L2)': []
}

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
   
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge (L2)': Ridge(alpha=0.01),
        'Lasso (L1)': Lasso(alpha=0.01),
        'ElasticNet (L1 + L2)': ElasticNet(alpha=0.01, l1_ratio = 0.5)
    }
    
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pr = model.predict(X_val_scaled)
        y_pred = (y_pr >= 0.5).astype(int)
        acc = accuracy_score(y_val, y_pred)
        fold_accuracy[name].append(acc)
        print(f'{name:<20} Accuracy: {acc:.4f}')

for name, accuracies in fold_accuracy.items():   
    print(f'{name:<20} Mean Accuracy: {np.mean(accuracies):.4f}')

Linear Regression    Accuracy: 0.8771
Ridge (L2)           Accuracy: 0.8771
Lasso (L1)           Accuracy: 0.8771
ElasticNet (L1 + L2) Accuracy: 0.8771
Linear Regression    Accuracy: 0.8315
Ridge (L2)           Accuracy: 0.8315
Lasso (L1)           Accuracy: 0.8427
ElasticNet (L1 + L2) Accuracy: 0.8371
Linear Regression    Accuracy: 0.8427
Ridge (L2)           Accuracy: 0.8427
Lasso (L1)           Accuracy: 0.8596
ElasticNet (L1 + L2) Accuracy: 0.8596
Linear Regression    Accuracy: 0.8652
Ridge (L2)           Accuracy: 0.8652
Lasso (L1)           Accuracy: 0.8652
ElasticNet (L1 + L2) Accuracy: 0.8652
Linear Regression    Accuracy: 0.8933
Ridge (L2)           Accuracy: 0.8933
Lasso (L1)           Accuracy: 0.8933
ElasticNet (L1 + L2) Accuracy: 0.9045
Linear Regression    Mean Accuracy: 0.8619
Ridge (L2)           Mean Accuracy: 0.8619
Lasso (L1)           Mean Accuracy: 0.8676
ElasticNet (L1 + L2) Mean Accuracy: 0.8687


In [7]:
#Kneighbors
n_neighbors_list = [3, 5, 7, 9, 11, 13, 15]
weights_list = ['uniform', 'distance']
metric_lists = ['euclidean', 'manhattan']
best_accuracy = 0.0
best_params = {}
for n in n_neighbors_list:
    for w in weights_list:
        for m in metric_lists:
            fold_accuracies = []
            for train_idx, val_idx in skf.split(X, y):
                X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
                X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_val_scaled = scaler.transform(X_val)
    
                model = KNeighborsClassifier(
                    n_neighbors = n,
                    weights = w,
                    metric = m
                )
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_val_scaled)
                acc = accuracy_score(y_val, y_pred)
                fold_accuracies.append(acc)
            mean_acc = np.mean(fold_accuracies)
            print(f'k = {n:<2} | weights = "{w:<8}" | metric = "{m:<9}" -> Mean Accuracy: {mean_acc:.4f}') 
            if mean_acc > best_accuracy:
                best_accuracy = mean_acc
                best_params = {
                    'n_neighbors': n,
                    'weights': w,
                    'metric': m
                }
print(f'Лучший результат: {best_accuracy}')
print(f'Параметры: {best_params}')


k = 3  | weights = "uniform " | metric = "euclidean" -> Mean Accuracy: 0.8406
k = 3  | weights = "uniform " | metric = "manhattan" -> Mean Accuracy: 0.8451
k = 3  | weights = "distance" | metric = "euclidean" -> Mean Accuracy: 0.8294
k = 3  | weights = "distance" | metric = "manhattan" -> Mean Accuracy: 0.8328
k = 5  | weights = "uniform " | metric = "euclidean" -> Mean Accuracy: 0.8608
k = 5  | weights = "uniform " | metric = "manhattan" -> Mean Accuracy: 0.8530
k = 5  | weights = "distance" | metric = "euclidean" -> Mean Accuracy: 0.8440
k = 5  | weights = "distance" | metric = "manhattan" -> Mean Accuracy: 0.8372
k = 7  | weights = "uniform " | metric = "euclidean" -> Mean Accuracy: 0.8530
k = 7  | weights = "uniform " | metric = "manhattan" -> Mean Accuracy: 0.8575
k = 7  | weights = "distance" | metric = "euclidean" -> Mean Accuracy: 0.8496
k = 7  | weights = "distance" | metric = "manhattan" -> Mean Accuracy: 0.8428
k = 9  | weights = "uniform " | metric = "euclidean" -> Mean Acc

In [8]:
#Decision Tree
fold_accuracy = []
for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = DecisionTreeClassifier()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8212
Accuracy: 0.7865
Accuracy: 0.8315
Accuracy: 0.8371
Accuracy: 0.8539
Mean Accuracy: 0.8260


In [9]:
#Random forest
fold_accuracy = []
best_accuracy = 0.0
best_param = 0
for n_est in range(100, 500, 10): 
    fold_accuracy = []
    for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        model = RandomForestClassifier(n_estimators = n_est)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
        acc = accuracy_score(y_val, y_pred)
        fold_accuracy.append(acc)
    mean_acc = np.mean(fold_accuracy)
    print(f'"{n_est:<2}" -> Mean accuracy = {mean_acc:.4f}')
    
    if mean_acc > best_accuracy:
        best_accuracy = mean_acc
        best_param = n_est

print(f'Лучший результат: {best_accuracy}')
print(f'Параметр: {best_param}')

"100" -> Mean accuracy = 0.8462
"110" -> Mean accuracy = 0.8473
"120" -> Mean accuracy = 0.8484
"130" -> Mean accuracy = 0.8473
"140" -> Mean accuracy = 0.8541
"150" -> Mean accuracy = 0.8451
"160" -> Mean accuracy = 0.8496
"170" -> Mean accuracy = 0.8529
"180" -> Mean accuracy = 0.8428
"190" -> Mean accuracy = 0.8529
"200" -> Mean accuracy = 0.8507
"210" -> Mean accuracy = 0.8473
"220" -> Mean accuracy = 0.8485
"230" -> Mean accuracy = 0.8473
"240" -> Mean accuracy = 0.8518
"250" -> Mean accuracy = 0.8496
"260" -> Mean accuracy = 0.8462
"270" -> Mean accuracy = 0.8518
"280" -> Mean accuracy = 0.8496
"290" -> Mean accuracy = 0.8484
"300" -> Mean accuracy = 0.8507
"310" -> Mean accuracy = 0.8496
"320" -> Mean accuracy = 0.8473
"330" -> Mean accuracy = 0.8518
"340" -> Mean accuracy = 0.8485
"350" -> Mean accuracy = 0.8484
"360" -> Mean accuracy = 0.8518
"370" -> Mean accuracy = 0.8518
"380" -> Mean accuracy = 0.8496
"390" -> Mean accuracy = 0.8518
"400" -> Mean accuracy = 0.8462
"410" ->

In [10]:
#XGBoost
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = XGBClassifier(
        n_estimators = 200,
        learning_rate = 0.05,
        max_depth = 4,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8827
Accuracy: 0.8539
Accuracy: 0.8483
Accuracy: 0.8820
Accuracy: 0.8708
Mean Accuracy: 0.8675


In [11]:
#CatBoost
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = CatBoostClassifier(
        iterations = 200,
        learning_rate = 0.05,
        depth = 4,
        verbose = 0,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8883
Accuracy: 0.8708
Accuracy: 0.8596
Accuracy: 0.8820
Accuracy: 0.8820
Mean Accuracy: 0.8765


In [12]:
#LGBMBoost
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = LGBMClassifier(
        n_estimators = 200,
        learning_rate = 0.05,
        num_leaves = 15,
        max_depth = 4,
        random_state = 42,
        verbose = -1
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8771
Accuracy: 0.8483
Accuracy: 0.8427
Accuracy: 0.8764
Accuracy: 0.8820
Mean Accuracy: 0.8653


In [13]:
#MLP
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = MLPClassifier(
        hidden_layer_sizes = (32, 16),
        activation = 'relu',
        max_iter = 1000,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8603
Accuracy: 0.8539
Accuracy: 0.8539
Accuracy: 0.8652
Accuracy: 0.8652
Mean Accuracy: 0.8597


In [14]:
#Logistic regression
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = LogisticRegression(
        max_iter = 1000,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8771
Accuracy: 0.8371
Accuracy: 0.8315
Accuracy: 0.8596
Accuracy: 0.8989
Mean Accuracy: 0.8608


In [15]:
#SVC
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val) 
    
    model = SVC(
        kernel = 'rbf',
        C = 1.0,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8883
Accuracy: 0.8483
Accuracy: 0.8483
Accuracy: 0.8596
Accuracy: 0.8708
Mean Accuracy: 0.8630


In [16]:
#GaussianNB
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = GaussianNB()
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.7989
Accuracy: 0.7978
Accuracy: 0.7921
Accuracy: 0.7921
Accuracy: 0.8034
Mean Accuracy: 0.7969


In [17]:
#DNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
class PoliksDNN(nn.Module):
    def __init__(self, input_dim, hidden_size, activat, drop_rate, batchnorm):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_size in hidden_size:
            layers.append(nn.Linear(in_dim, h_size))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h_size))
            layers.append(activat)
            if drop_rate > 0:
                layers.append(nn.Dropout(drop_rate))
            in_dim = h_size
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)
    
def experim(
    hidden_size = [32, 16],
    activat = nn.ReLU(),
    drop_rate = 0.2,
    batchnorm = True,
    optimiz = optim.Adam,
    lr = 0.01,
    batch_size = 32,
    epochs = 20,
    sched = True
):
    X_tensor = torch.tensor(X.values, dtype = torch.float32)
    y_tensor = torch.tensor(y.values, dtype = torch.float32).unsqueeze(1)

    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size = batch_size, shuffle = True)
    
    model = PoliksDNN(X.shape[1], hidden_size, activat, drop_rate, batchnorm)
    loss_func = nn.BCEWithLogitsLoss()
    optimizer = optimiz(model.parameters(), lr = lr)
    
    if sched:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = epochs)

    for epoch in range(epochs):
        model.train()
        for x_, y_ in loader:
            optimizer.zero_grad()
            out = model(x_)
            loss = loss_func(out, y_)
            loss.backward()
            optimizer.step()
        if sched:
            scheduler.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_tensor)).numpy().flatten()
        preds = (probs > 0.5).astype(int)
        y_true = y_tensor.numpy().flatten()
        
        acc = accuracy_score(y_true, preds)
        prec = precision_score(y_true, preds, zero_division=0)
        rec = recall_score(y_true, preds, zero_division=0)
        f1 = f1_score(y_true, preds, zero_division=0)
        auc = roc_auc_score(y_true, probs)
        cm = confusion_matrix(y_true, preds)

        print(f"Accuracy: {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall: {rec:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(f"ROC-AUC: {auc:.4f}")
        print(f"Confusion Matrix:\n{cm}")
    return acc, model

In [18]:
acc1, model1 = experim(hidden_size = [32, 16], activat = nn.ReLU(), optimiz = optim.Adam)
print(f'Exp1: Accuracy = {acc1:.4f}')

Accuracy: 0.8934
Precision: 0.9364
Recall: 0.7749
F1-Score: 0.8480
ROC-AUC: 0.9416
Confusion Matrix:
[[531  18]
 [ 77 265]]
Exp1: Accuracy = 0.8934


In [19]:
acc2, model2 = experim(hidden_size = [32, 16], activat = nn.Tanh(), optimiz = optim.Adam)
print(f'Exp2: Accuracy = {acc2:.4f}')

Accuracy: 0.8833
Precision: 0.8993
Recall: 0.7836
F1-Score: 0.8375
ROC-AUC: 0.9279
Confusion Matrix:
[[519  30]
 [ 74 268]]
Exp2: Accuracy = 0.8833


In [20]:
acc3, model3 = experim(hidden_size = [32, 16], activat = nn.Tanh(), optimiz = optim.SGD)
print(f'Exp3: Accuracy = {acc3:.4f}')

Accuracy: 0.7363
Precision: 0.6766
Recall: 0.5994
F1-Score: 0.6357
ROC-AUC: 0.7715
Confusion Matrix:
[[451  98]
 [137 205]]
Exp3: Accuracy = 0.7363


In [21]:
acc4, model4 = experim(hidden_size = [32, 16], activat = nn.ReLU(), optimiz = optim.SGD)
print(f'Exp4: Accuracy = {acc4:.4f}')

Accuracy: 0.6880
Precision: 0.6975
Recall: 0.3304
F1-Score: 0.4484
ROC-AUC: 0.7129
Confusion Matrix:
[[500  49]
 [229 113]]
Exp4: Accuracy = 0.6880


In [22]:
acc5, model5 = experim(hidden_size = [32, 16], activat = nn.ReLU(), optimiz = optim.Adam, epochs = 30)
print(f'Exp5: Accuracy = {acc5:.4f}')

Accuracy: 0.8923
Precision: 0.9184
Recall: 0.7895
F1-Score: 0.8491
ROC-AUC: 0.9431
Confusion Matrix:
[[525  24]
 [ 72 270]]
Exp5: Accuracy = 0.8923


In [23]:
acc6, model6 = experim(hidden_size = [64, 32, 16], activat = nn.ReLU(), drop_rate = 0.4, optimiz = optim.Adam, epochs = 30)
print(f'Exp6: Accuracy = {acc6:.4f}')

Accuracy: 0.8833
Precision: 0.9161
Recall: 0.7661
F1-Score: 0.8344
ROC-AUC: 0.9385
Confusion Matrix:
[[525  24]
 [ 80 262]]
Exp6: Accuracy = 0.8833


In [24]:
acc7, model7 = experim(hidden_size = [64, 32, 16], activat = nn.ReLU(), drop_rate = 0.3, optimiz = optim.Adam, epochs = 30)
print(f'Exp7: Accuracy = {acc7:.4f}')

Accuracy: 0.8945
Precision: 0.9247
Recall: 0.7895
F1-Score: 0.8517
ROC-AUC: 0.9410
Confusion Matrix:
[[527  22]
 [ 72 270]]
Exp7: Accuracy = 0.8945


In [25]:
acc8, model8 = experim(hidden_size = [64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, epochs = 30)
print(f'Exp8: Accuracy = {acc8:.4f}')

Accuracy: 0.8979
Precision: 0.9226
Recall: 0.8012
F1-Score: 0.8576
ROC-AUC: 0.9449
Confusion Matrix:
[[526  23]
 [ 68 274]]
Exp8: Accuracy = 0.8979


In [26]:
acc9, model9 = experim(hidden_size = [64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, epochs = 30)
print(f'Exp9: Accuracy = {acc9:.4f}')

Accuracy: 0.8934
Precision: 0.9076
Recall: 0.8041
F1-Score: 0.8527
ROC-AUC: 0.9449
Confusion Matrix:
[[521  28]
 [ 67 275]]
Exp9: Accuracy = 0.8934


In [27]:
acc10, model10 = experim(hidden_size = [32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, epochs = 30)
print(f'Exp10: Accuracy = {acc10:.4f}')

Accuracy: 0.8934
Precision: 0.9103
Recall: 0.8012
F1-Score: 0.8523
ROC-AUC: 0.9413
Confusion Matrix:
[[522  27]
 [ 68 274]]
Exp10: Accuracy = 0.8934


In [28]:
acc11, model11 = experim(hidden_size = [32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, epochs = 30, sched = False)
print(f'Exp11: Accuracy = {acc11:.4f}')

Accuracy: 0.8923
Precision: 0.9212
Recall: 0.7865
F1-Score: 0.8486
ROC-AUC: 0.9411
Confusion Matrix:
[[526  23]
 [ 73 269]]
Exp11: Accuracy = 0.8923


In [29]:
acc12, model12 = experim(hidden_size = [32, 16], activat = nn.ReLU(), drop_rate = 0.2, batchnorm = False, optimiz = optim.Adam, epochs = 30)
print(f'Exp12: Accuracy = {acc12:.4f}')

Accuracy: 0.8855
Precision: 0.9412
Recall: 0.7485
F1-Score: 0.8339
ROC-AUC: 0.9295
Confusion Matrix:
[[533  16]
 [ 86 256]]
Exp12: Accuracy = 0.8855


In [30]:
acc13, model13 = experim(hidden_size = [64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, batchnorm = False, optimiz = optim.Adam, epochs = 30)
print(f'Exp13: Accuracy = {acc13:.4f}')

Accuracy: 0.8844
Precision: 0.9314
Recall: 0.7544
F1-Score: 0.8336
ROC-AUC: 0.9347
Confusion Matrix:
[[530  19]
 [ 84 258]]
Exp13: Accuracy = 0.8844


In [31]:
acc14, model14 = experim(hidden_size = [32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 64, epochs = 30)
print(f'Exp14: Accuracy = {acc14:.4f}')

Accuracy: 0.8934
Precision: 0.9186
Recall: 0.7924
F1-Score: 0.8509
ROC-AUC: 0.9429
Confusion Matrix:
[[525  24]
 [ 71 271]]
Exp14: Accuracy = 0.8934


In [32]:
acc15, model15 = experim(hidden_size = [32, 16], activat = nn.ReLU(), drop_rate = 0.2, batchnorm = False, optimiz = optim.Adam, batch_size = 64, epochs = 30)
print(f'Exp15: Accuracy = {acc15:.4f}')

Accuracy: 0.8788
Precision: 0.9239
Recall: 0.7456
F1-Score: 0.8252
ROC-AUC: 0.9259
Confusion Matrix:
[[528  21]
 [ 87 255]]
Exp15: Accuracy = 0.8788


In [33]:
acc16, model16 = experim(hidden_size = [64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 64, epochs = 30)
print(f'Exp16: Accuracy = {acc16:.4f}')

Accuracy: 0.9001
Precision: 0.9148
Recall: 0.8158
F1-Score: 0.8624
ROC-AUC: 0.9465
Confusion Matrix:
[[523  26]
 [ 63 279]]
Exp16: Accuracy = 0.9001


In [34]:
acc17, model17 = experim(hidden_size = [128, 64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 64, epochs = 30)
print(f'Exp17: Accuracy = {acc17:.4f}')

Accuracy: 0.8979
Precision: 0.9115
Recall: 0.8129
F1-Score: 0.8594
ROC-AUC: 0.9471
Confusion Matrix:
[[522  27]
 [ 64 278]]
Exp17: Accuracy = 0.8979


In [35]:
acc18, model18 = experim(hidden_size = [128, 64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 128, epochs = 30)
print(f'Exp18: Accuracy = {acc18:.4f}')

Accuracy: 0.8956
Precision: 0.9136
Recall: 0.8041
F1-Score: 0.8554
ROC-AUC: 0.9466
Confusion Matrix:
[[523  26]
 [ 67 275]]
Exp18: Accuracy = 0.8956


In [36]:
acc19, model19 = experim(hidden_size = [256, 128, 64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 128, epochs = 30)
print(f'Exp19: Accuracy = {acc19:.4f}')

Accuracy: 0.9068
Precision: 0.9360
Recall: 0.8129
F1-Score: 0.8701
ROC-AUC: 0.9496
Confusion Matrix:
[[530  19]
 [ 64 278]]
Exp19: Accuracy = 0.9068


In [37]:
acc20, model20 = experim(hidden_size = [128, 64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 64, epochs = 30)
print(f'Exp20: Accuracy = {acc20:.4f}')

Accuracy: 0.8990
Precision: 0.9172
Recall: 0.8099
F1-Score: 0.8602
ROC-AUC: 0.9492
Confusion Matrix:
[[524  25]
 [ 65 277]]
Exp20: Accuracy = 0.8990


In [38]:
acc21, model21 = experim(hidden_size = [128, 64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 32, epochs = 30)
print(f'Exp21: Accuracy = {acc21:.4f}')

Accuracy: 0.8956
Precision: 0.9082
Recall: 0.8099
F1-Score: 0.8563
ROC-AUC: 0.9453
Confusion Matrix:
[[521  28]
 [ 65 277]]
Exp21: Accuracy = 0.8956


In [39]:
acc22, model22 = experim(hidden_size = [128, 64, 32, 16], activat = nn.GELU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 64, epochs = 50)
print(f'Exp22: Accuracy = {acc22:.4f}')

Accuracy: 0.9035
Precision: 0.9295
Recall: 0.8099
F1-Score: 0.8656
ROC-AUC: 0.9479
Confusion Matrix:
[[528  21]
 [ 65 277]]
Exp22: Accuracy = 0.9035


In [40]:
import numpy as np

scores = []
for i in range(5):
    acc, model = experim(
        hidden_size=[128, 64, 32, 16],
        activat=nn.GELU(),
        drop_rate=0.2,
        batch_size=64,
        epochs=50
    )
    scores.append(acc)
    
mean_acc = np.mean(scores)
std_acc = np.std(scores)

print("\n" + "="*35)
print(f"Точности прогонов: {[round(s, 4) for s in scores]}")
print(f"ИТОГ: Accuracy = {mean_acc:.4f} ± {std_acc:.4f}")
print("="*35)

Accuracy: 0.9046
Precision: 0.9132
Recall: 0.8304
F1-Score: 0.8698
ROC-AUC: 0.9509
Confusion Matrix:
[[522  27]
 [ 58 284]]
Accuracy: 0.9057
Precision: 0.9388
Recall: 0.8070
F1-Score: 0.8679
ROC-AUC: 0.9510
Confusion Matrix:
[[531  18]
 [ 66 276]]
Accuracy: 0.9057
Precision: 0.9243
Recall: 0.8216
F1-Score: 0.8700
ROC-AUC: 0.9531
Confusion Matrix:
[[526  23]
 [ 61 281]]
Accuracy: 0.9136
Precision: 0.9288
Recall: 0.8392
F1-Score: 0.8817
ROC-AUC: 0.9505
Confusion Matrix:
[[527  22]
 [ 55 287]]
Accuracy: 0.9046
Precision: 0.9298
Recall: 0.8129
F1-Score: 0.8674
ROC-AUC: 0.9512
Confusion Matrix:
[[528  21]
 [ 64 278]]

Точности прогонов: [0.9046, 0.9057, 0.9057, 0.9136, 0.9046]
ИТОГ: Accuracy = 0.9068 ± 0.0034


In [41]:
acc23, model23 = experim(hidden_size = [128, 64, 32, 16], activat = nn.ReLU(), drop_rate = 0.2, optimiz = optim.Adam, batch_size = 64, epochs = 30)
print(f'Exp23: Accuracy = {acc23:.4f}')

Accuracy: 0.9046
Precision: 0.9213
Recall: 0.8216
F1-Score: 0.8686
ROC-AUC: 0.9497
Confusion Matrix:
[[525  24]
 [ 61 281]]
Exp23: Accuracy = 0.9046


In [42]:
#GridSearchCV
import warnings
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

X = train_clean.drop(columns=['Survived'])
y = train_clean['Survived']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def create_pipe(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

experiments = {
    'LogisticRegression': (
        create_pipe(LogisticRegression(max_iter=1000, random_state=42)),
        {
            'model__C': [0.1, 1.0, 10.0],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear']
        }
    ),
    'RidgeClassifier': (
        create_pipe(RidgeClassifier(random_state=42)),
        {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    ),
    'SVC': (
        create_pipe(SVC(probability=True, random_state=42)),
        {
            'model__C': [0.1, 1.0, 5.0],
            'model__kernel': ['rbf']
        }
    ),
    'KNN': (
        create_pipe(KNeighborsClassifier()),
        {
            'model__n_neighbors': [5, 7, 9],
            'model__weights': ['uniform', 'distance']
        }
    ),
    'RandomForest': (
        create_pipe(RandomForestClassifier(random_state=42, n_jobs=-1)),
        {
            'model__n_estimators': [100, 200],
            'model__max_depth': [3, 5, 7]
        }
    ),
    'XGBoost': (
        create_pipe(XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)),
        {
            'model__n_estimators': [100],
            'model__learning_rate': [0.03, 0.1],
            'model__max_depth': [3, 5]
        }
    ),
    'LightGBM': (
        create_pipe(LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1)),
        {
            'model__n_estimators': [100],
            'model__learning_rate': [0.03, 0.1],
            'model__max_depth': [3, 5]
        }
    ),
    'CatBoost': (
        create_pipe(CatBoostClassifier(random_state=42, verbose=0, thread_count=-1)),
        {
            'model__iterations': [150],
            'model__learning_rate': [0.03, 0.1],
            'model__depth': [3, 5]
        }
    )
}

best_estimators = {}
res_summary = []

for name, (pipeline, param_grid) in experiments.items():
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=1 
    )
    grid.fit(X, y)
    
    best_estimators[name] = grid.best_estimator_
    res_summary.append({
        'Model': name,
        'Best Accuracy': grid.best_score_,
        'Best Params': grid.best_params_
    })
    print(f"[{name}] Best Accuracy: {grid.best_score_:.4f}")

results_dt = pd.DataFrame(res_summary).sort_values(by='Best Accuracy', ascending=False)
pd.set_option('display.max_colwidth', None)
display(results_dt)



[LogisticRegression] Best Accuracy: 0.8709
[RidgeClassifier] Best Accuracy: 0.8653
[SVC] Best Accuracy: 0.8630
[KNN] Best Accuracy: 0.8608
[RandomForest] Best Accuracy: 0.8698
[XGBoost] Best Accuracy: 0.8732
[LightGBM] Best Accuracy: 0.8676
[CatBoost] Best Accuracy: 0.8799


,Model,Best Accuracy,Best Params
7,CatBoost,0.879876,"{'model__depth': 5, 'model__iterations': 150, 'model__learning_rate': 0.1}"
5,XGBoost,0.873166,"{'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 100}"
0,LogisticRegression,0.870918,"{'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'liblinear'}"
4,RandomForest,0.869801,"{'model__max_depth': 5, 'model__n_estimators': 100}"
6,LightGBM,0.867554,"{'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__n_estimators': 100}"
1,RidgeClassifier,0.865307,{'model__alpha': 10.0}
2,SVC,0.863047,"{'model__C': 1.0, 'model__kernel': 'rbf'}"
3,KNN,0.860819,"{'model__n_neighbors': 5, 'model__weights': 'uniform'}"
